# MRI Embedding Extraction

**Pipeline position:** CNN (trained) → **Extract MRI Embeddings** → Merge with Clinical → GCN

This notebook:
1. Rebuilds the inventory with the same 3-class label merge used during training
2. Loads `best_resnet18.pth` and strips the classification head → 512-dim embedding extractor
3. Runs **all images** (train + val + test) through the extractor — no augmentation
4. Aggregates per-subject: **mean-pools** all slice embeddings for a subject → one 512-dim vector
5. Saves `mri_embeddings.csv` — one row per subject, columns `subject_id`, `label`, `emb_0` … `emb_511`


In [1]:
!pip install -q kaggle

In [2]:
import os

os.environ["KAGGLE_API_TOKEN"] = "KGAT_283606f7db27511d5bedbdc1c43c0695"

In [3]:
!kaggle datasets list -s oasis

ref                                                            title                                                    size  lastUpdated                 downloadCount  voteCount  usabilityRating  
-------------------------------------------------------------  -----------------------------------------------  ------------  --------------------------  -------------  ---------  ---------------  
ninadaithal/imagesoasis                                        OASIS Alzheimer's Detection                        1322017985  2023-06-18 13:23:08.663000          26353        162  0.875            
federicoseijo/oasis-discography                                Oasis discography                                       39039  2021-09-25 13:02:59.570000            286         16  0.9411765        
jboysen/mri-and-alzheimers                                     MRI and Alzheimers                                      12924  2017-08-16 17:18:10.663000          39329        556  0.85294116       
shreyanmoh

In [4]:
!kaggle datasets download -d ninadaithal/imagesoasis

Dataset URL: https://www.kaggle.com/datasets/ninadaithal/imagesoasis
License(s): apache-2.0
100% 1.23G/1.23G [00:09<00:00, 137MB/s]



In [5]:
!unzip -q imagesoasis.zip -d MRI_DATA

In [6]:
import os

for folder in os.listdir("MRI_DATA"):
    print(folder)

Data


In [7]:
from torchvision.datasets import ImageFolder

dataset = ImageFolder(
    root="MRI_DATA/Data"
)

print(dataset.classes)
print(len(dataset))

['Mild Dementia', 'Moderate Dementia', 'Non Demented', 'Very mild Dementia']
86437


## 1. Imports & Setup

In [8]:
import os
import re
import random
import numpy as np
import pandas as pd

from PIL import Image

import torch
import torch.nn as nn

from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


Device: cuda


## 2. Config

In [9]:
from google.colab import drive
drive.mount("/content/drive")

import json

PROJECT_DIR = "/content/drive/MyDrive/alzheimer_project"
DATA_DIR    = "/content/MRI_DATA/Data"
MODEL_PATH  = f"{PROJECT_DIR}/best_resnet18.pth"
OUTPUT_CSV  = f"{PROJECT_DIR}/mri_embeddings.csv"

BATCH_SIZE  = 64
NUM_WORKERS = 2

# Load class mapping saved by CNN notebook
with open(f"{PROJECT_DIR}/class_mapping.json") as f:
    mapping = json.load(f)

CLASS_NAMES  = mapping["CLASS_NAMES"]
CLASS_TO_IDX = mapping["CLASS_TO_IDX"]
LABEL_MERGE  = mapping["LABEL_MERGE"]

print("CLASS_NAMES :", CLASS_NAMES)
print("CLASS_TO_IDX:", CLASS_TO_IDX)
print("MODEL_PATH  :", MODEL_PATH)
print("OUTPUT_CSV  :", OUTPUT_CSV)

Mounted at /content/drive
CLASS_NAMES : ['Dementia', 'Non Demented', 'Very mild Dementia']
CLASS_TO_IDX: {'Dementia': 0, 'Non Demented': 1, 'Very mild Dementia': 2}
MODEL_PATH  : /content/drive/MyDrive/alzheimer_project/best_resnet18.pth
OUTPUT_CSV  : /content/drive/MyDrive/alzheimer_project/mri_embeddings.csv


## 3. Build Full Inventory (all subjects, no split)

In [10]:
import re
all_records = []
subject_pattern = re.compile(r"(OAS\d+_\d+)", re.IGNORECASE)

for class_name in os.listdir(DATA_DIR):
    class_path = os.path.join(DATA_DIR, class_name)
    if not os.path.isdir(class_path):
        continue

    merged_label = LABEL_MERGE.get(class_name, class_name)

    for fname in os.listdir(class_path):
        fpath = os.path.join(class_path, fname)
        match = subject_pattern.search(fname)
        subject_id = match.group(1) if match else fname

        all_records.append({
            "subject_id": subject_id,
            "image_path": fpath,
            "label":      merged_label,
        })

inventory_df = pd.DataFrame(all_records)

print(f"Total images : {len(inventory_df):,}")
print(f"Unique subjects: {inventory_df['subject_id'].nunique()}")
print()
print(inventory_df['label'].value_counts())


Total images : 86,437
Unique subjects: 347

label
Non Demented          67222
Very mild Dementia    13725
Dementia               5490
Name: count, dtype: int64


## 4. Dataset — Inference Transform Only (no augmentation)

In [11]:
# No augmentation — we want deterministic, stable embeddings
infer_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


class MRIEmbedDataset(Dataset):
    """Returns (image_tensor, label_idx, subject_id) for every slice."""

    def __init__(self, dataframe, transform):
        self.df        = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        image = self.transform(image)
        label = CLASS_TO_IDX[row["label"]]
        return image, label, row["subject_id"]


full_dataset = MRIEmbedDataset(inventory_df, infer_transform)

full_loader = DataLoader(
    full_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

print(f"Batches: {len(full_loader)} × batch_size={BATCH_SIZE}")


Batches: 1351 × batch_size=64


## 5. Load Trained ResNet18 — Strip Classification Head

ResNet18 architecture:
```
... → layer4 → avgpool → [fc: 512→3]  ← remove this
                  ↑
            512-dim vector  ← this is the embedding
```
We replace `fc` with `nn.Identity()` so the forward pass returns the 512-dim pooled vector.


In [12]:
# Rebuild model with same architecture as training
model = models.resnet18(weights=None)      # no pretrained weights — we load our own
model.fc = nn.Linear(model.fc.in_features, 3)

model.load_state_dict(
    torch.load(MODEL_PATH, map_location=DEVICE)
)
print("Loaded:", MODEL_PATH)

# Strip the classification head → 512-dim embedding extractor
model.fc = nn.Identity()
model     = model.to(DEVICE)
model.eval()

print("Embedding dimension: 512")
print("Model set to eval mode — no dropout / batchnorm in train mode")


Loaded: /content/drive/MyDrive/alzheimer_project/best_resnet18.pth
Embedding dimension: 512
Model set to eval mode — no dropout / batchnorm in train mode


## 6. Extract Slice-Level Embeddings

In [13]:
slice_embeddings = []   # list of dicts: subject_id, label_idx, emb vector

with torch.no_grad():
    for batch_idx, (images, labels, subject_ids) in enumerate(full_loader):

        images = images.to(DEVICE)
        embs   = model(images)          # (batch, 512)
        embs   = embs.cpu().numpy()

        for i in range(len(subject_ids)):
            slice_embeddings.append({
                "subject_id": subject_ids[i],
                "label_idx":  labels[i].item(),
                "embedding":  embs[i],   # numpy array (512,)
            })

        if (batch_idx + 1) % 50 == 0:
            print(f"  Processed {(batch_idx+1)*BATCH_SIZE:,} / {len(full_dataset):,} slices")

print(f"\nDone. Total slice embeddings: {len(slice_embeddings):,}")


  Processed 3,200 / 86,437 slices
  Processed 6,400 / 86,437 slices
  Processed 9,600 / 86,437 slices
  Processed 12,800 / 86,437 slices
  Processed 16,000 / 86,437 slices
  Processed 19,200 / 86,437 slices
  Processed 22,400 / 86,437 slices
  Processed 25,600 / 86,437 slices
  Processed 28,800 / 86,437 slices
  Processed 32,000 / 86,437 slices
  Processed 35,200 / 86,437 slices
  Processed 38,400 / 86,437 slices
  Processed 41,600 / 86,437 slices
  Processed 44,800 / 86,437 slices
  Processed 48,000 / 86,437 slices
  Processed 51,200 / 86,437 slices
  Processed 54,400 / 86,437 slices
  Processed 57,600 / 86,437 slices
  Processed 60,800 / 86,437 slices
  Processed 64,000 / 86,437 slices
  Processed 67,200 / 86,437 slices
  Processed 70,400 / 86,437 slices
  Processed 73,600 / 86,437 slices
  Processed 76,800 / 86,437 slices
  Processed 80,000 / 86,437 slices
  Processed 83,200 / 86,437 slices
  Processed 86,400 / 86,437 slices

Done. Total slice embeddings: 86,437


## 7. Mean-Pool Slices → One Vector per Subject

Each subject has many MRI slices. We average all slice embeddings per subject.
This gives one 512-dim vector per subject — the unit the GCN will operate on.


In [14]:
from collections import defaultdict

subject_emb_accum  = defaultdict(list)   # subject_id → [emb, emb, ...]
subject_label      = {}                   # subject_id → label_idx (consistent)

for record in slice_embeddings:
    sid = record["subject_id"]
    subject_emb_accum[sid].append(record["embedding"])
    subject_label[sid] = record["label_idx"]

# Build mean-pooled embedding per subject
rows = []
for sid, emb_list in subject_emb_accum.items():
    mean_emb = np.mean(emb_list, axis=0)   # (512,)
    label_idx = subject_label[sid]
    label_name = CLASS_NAMES[label_idx]

    row = {"subject_id": sid, "label": label_name}
    for dim, val in enumerate(mean_emb):
        row[f"emb_{dim}"] = val
    rows.append(row)

embeddings_df = pd.DataFrame(rows)

print(f"Subjects: {len(embeddings_df)}")
print(f"Columns : subject_id, label + {len(mean_emb)} embedding dims")
print()
print(embeddings_df['label'].value_counts())


Subjects: 347
Columns : subject_id, label + 512 embedding dims

label
Non Demented          266
Very mild Dementia     58
Dementia               23
Name: count, dtype: int64


## 8. Sanity Checks

In [15]:
emb_cols = [c for c in embeddings_df.columns if c.startswith("emb_")]

# 1. No NaNs
nan_count = embeddings_df[emb_cols].isna().sum().sum()
print(f"NaN values in embeddings: {nan_count}")

# 2. Embedding dimension
print(f"Embedding dimension: {len(emb_cols)}  (expected 512)")

# 3. L2 norm of first few subjects — should be non-zero, similar magnitudes
sample = embeddings_df[emb_cols].values[:5]
norms  = np.linalg.norm(sample, axis=1)
print(f"L2 norms of first 5 subjects: {norms.round(3)}")

# 4. Preview
print()
embeddings_df[['subject_id', 'label'] + emb_cols[:5]].head()


NaN values in embeddings: 0
Embedding dimension: 512  (expected 512)
L2 norms of first 5 subjects: [32.215 14.579 26.173 14.393 14.827]



,subject_id,label,emb_0,emb_1,emb_2,emb_3,emb_4
0,OAS1_0351,Dementia,2.166160,1.735636,1.416839,0.433220,0.828611
1,OAS1_0308,Dementia,0.638422,0.964953,0.859017,1.198233,0.738115
2,OAS1_0053,Dementia,1.749306,1.919430,1.905636,1.816800,0.898333
3,OAS1_0223,Dementia,0.814133,0.930930,1.099241,1.426746,0.522730
4,OAS1_0373,Dementia,0.519155,0.637669,0.420800,0.719033,0.297324


## 9. Save Embeddings

In [16]:
embeddings_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved: {OUTPUT_CSV}")
print(f"Shape: {embeddings_df.shape}")
print(f"  Rows: {len(embeddings_df)} subjects")
print(f"  Cols: subject_id + label + 512 embedding dims")


Saved: /content/drive/MyDrive/alzheimer_project/mri_embeddings.csv
Shape: (347, 514)
  Rows: 347 subjects
  Cols: subject_id + label + 512 embedding dims


## 10. Next Step: Merge with Clinical Features

`mri_embeddings.csv` has one row per subject.  
In the next notebook, load the OASIS clinical CSV and merge on `subject_id`:

```python
clinical_df    = pd.read_csv('oasis_cross-sectional.csv')
mri_df         = pd.read_csv('mri_embeddings.csv')

# Align subject IDs (OASIS uses 'ID' column)
clinical_df    = clinical_df.rename(columns={'ID': 'subject_id'})
fusion_df      = mri_df.merge(clinical_df, on='subject_id', how='inner')
```

The `fusion_df` is what feeds the GCN node feature matrix.


In [17]:
import os
print(os.path.exists(f"{PROJECT_DIR}/mri_embeddings.csv"))
# Should print: True

True


In [18]:
from google.colab.files import download
download(f"{PROJECT_DIR}/mri_embeddings.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>